# Chapter 7 — 데이터 분석 방법론과 환경 구성 (실습)

## 실습 목표
1. Colab 세션 + Drive 마운트
2. CICIDS2017 mini subset 로드 (합성 mirror)
3. 라벨 분포·기본 통계 확인
4. 본 실습이 채택한 분석 프로세스 다이어그램에 자신의 시나리오 매핑

---
## Step 0. 환경 점검

In [ ]:
# 파트별 노트북은 같은 종류의 공통 데이터·모델 작업 폴더를 사용합니다.
from pathlib import Path as _BookPath
import os as _book_os
_book_cwd = _BookPath.cwd()
if (_book_cwd.name in {"part1", "part2", "part3", "part4"}
        and _book_cwd.parent.name in {"instructor", "exercises", "solutions"}):
    _book_os.chdir(_book_cwd.parent)


In [1]:
# 첫 사용시에만 주석 해제하여 실행 (Colab/venv 환경에 따라 일부는 이미 설치됨)
# !pip install -q pandas
# !pip install -q numpy
# !pip install -q scikit-learn

In [2]:
import sys, platform
import numpy as np
import pandas as pd
import sklearn
print(f'Python   {sys.version.split()[0]}  /  {platform.system()}')
print(f'numpy    {np.__version__}')
print(f'pandas   {pd.__version__}')
print(f'sklearn  {sklearn.__version__}')

Python   3.12.5  /  Linux
numpy    2.4.4
pandas   3.0.2
sklearn  1.8.0


In [3]:
# Google Drive mount — Colab 환경에서만 (로컬에서는 건너뜀)
try:
    from google.colab import drive
    drive.mount('/content/gdrive')
    DATA_DIR = '/content/gdrive/My Drive/AI보안_네트워크/data/'
except ImportError:
    DATA_DIR = './data/'
    print('로컬 환경 — Drive mount 건너뜀')

로컬 환경 — Drive mount 건너뜀


---
## Step 1. CICIDS2017 mini subset 로드

In [4]:
# 합성 mini subset (ch04 과 동일한 합성 구조)
rng = np.random.default_rng(42)
n_benign, n_ddos, n_scan = 16000, 2300, 1200

def make_rows(n, label, dur_range, pkt_range, syn_range):
    return pd.DataFrame({
        'Source IP': rng.choice([f'10.0.0.{i}' for i in range(1,50)], size=n),
        'Destination Port': rng.choice([80, 443, 53, 22] if label=='BENIGN'
                                       else list(range(1,1024)), size=n),
        'Flow Duration': rng.uniform(*dur_range, size=n),
        'Total Fwd Packets': rng.integers(*pkt_range, size=n),
        'SYN Flag Count': rng.uniform(*syn_range, size=n),
        'Label': label,
    })

df = pd.concat([
    make_rows(n_benign, 'BENIGN',   (1000, 1_000_000), (5, 100),  (0, 0.05)),
    make_rows(n_ddos,   'DDoS',     (10, 5000),         (500, 5000), (0.4, 0.7)),
    make_rows(n_scan,   'PortScan', (1, 200),           (1, 5),     (0.85, 1.0)),
], ignore_index=True).sample(frac=1, random_state=0).reset_index(drop=True)

print(f'shape: {df.shape}')
df.head()

shape: (19500, 6)


,Source IP,Destination Port,Flow Duration,Total Fwd Packets,SYN Flag Count,Label
0,10.0.0.35,80,48068.615060,77,0.018195,BENIGN
1,10.0.0.28,53,961721.752453,30,0.030678,BENIGN
2,10.0.0.10,443,446554.504113,10,0.017566,BENIGN
3,10.0.0.24,998,15.809071,4,0.917880,PortScan
4,10.0.0.9,372,4960.320817,876,0.611898,DDoS


---
## Step 2. 라벨 분포 + 기본 통계

In [5]:
print('--- 라벨 분포 ---')
print(df['Label'].value_counts(normalize=True).mul(100).round(2))
print('\n--- 라벨별 통계 ---')
df.groupby('Label')[['Flow Duration', 'Total Fwd Packets', 'SYN Flag Count']].describe().T

--- 라벨 분포 ---
Label
BENIGN      82.05
DDoS        11.79
PortScan     6.15
Name: proportion, dtype: float64

--- 라벨별 통계 ---


Label                           BENIGN         DDoS     PortScan
Flow Duration     count   16000.000000  2300.000000  1200.000000
                  mean   503384.455651  2533.629664    99.978206
                  std    288524.220389  1447.879705    57.081893
                  min      1019.220277    10.354430     1.043929
                  25%    253476.786841  1308.654238    50.772603
                  50%    503893.849718  2526.797271    98.742910
                  75%    754695.954366  3805.556607   147.360041
                  max    999982.505954  4998.302300   199.825861
Total Fwd Packets count   16000.000000  2300.000000  1200.000000
                  mean       51.999250  2769.676087     2.483333
                  std        27.457777  1295.321096     1.118376
                  min         5.000000   500.000000     1.000000
                  25%        28.000000  1650.500000     1.000000
                  50%        52.000000  2771.500000     2.000000
                  75%        76.000000  3901.750000     3.000000
                  max        99.000000  4994.000000     4.000000
SYN Flag Count    count   16000.000000  2300.000000  1200.000000
                  mean        0.024969     0.549818     0.922044
                  std         0.014396     0.086345     0.044012
                  min         0.000003     0.400004     0.850347
                  25%         0.012534     0.476913     0.881170
                  50%         0.025021     0.551181     0.921420
                  75%         0.037330     0.625193     0.960239
                  max         0.050000     0.699610     0.999948

---
## Step 3. 분석 프로세스 매핑 — 빅데이터 방법론을 본인 시나리오에

In [6]:
process = pd.DataFrame([
    {'단계': 'Planning',   '내용': '오늘의 데이터(CICIDS2017)로 PortScan vs DDoS vs BENIGN 분류'},
    {'단계': 'Preparing',  '내용': 'pandas 로 로드, 결측치 확인 (ch08 본격)'},
    {'단계': 'Analyzing',  '내용': '전처리·EDA·모델링·평가 (ch09~ch16)'},
    {'단계': 'Developing', '내용': 'LLM·RAG·Agent 통합 (ch17~ch20)'},
    {'단계': 'Deploying',  '내용': '실습 코드 → 실 운영 환경 이식'},
])
process

,단계,내용
0,Planning,오늘의 데이터(CICIDS2017)로 PortScan vs DDoS vs BENIG...
1,Preparing,"pandas 로 로드, 결측치 확인 (ch08 본격)"
2,Analyzing,전처리·EDA·모델링·평가 (ch09~ch16)
3,Developing,LLM·RAG·Agent 통합 (ch17~ch20)
4,Deploying,실습 코드 → 실 운영 환경 이식


---
## 정리

- Colab + Drive 마운트가 본 실습 표준 환경.
- 한 데이터프레임만 있어도 라벨 분포·기본 통계·간단한 통계 차이가 즉시 보인다.
- 다음 챕터에서 본격 EDA·전처리로 진입한다.